# ALM observado — Colombia: depósitos, cartera y sensibilidad de tasas

Laboratorio exploratorio con fuentes **observadas** de SFC y Banco de la República almacenadas en Master_DATOS. Los escenarios son **hipotéticos**; el notebook no usa datos sintéticos como sustituto de datos reales.

\(L_t=C_t/(D_{corr,t}+D_{ah,t}+CDT_t)\) y \(\Delta Coste=CDT_t\,(\Delta r/10^4)\,\theta\), donde \(\theta\) es *pass-through supuesto*. No son indicadores regulatorios ni costos contractuales observados.

In [ ]:
from pathlib import Path
import os, sys
import matplotlib.pyplot as plt
candidates=[Path.cwd(), *Path.cwd().parents]
base=next((p for p in candidates if (p/'04-quant-strategy-and-products/treasury-and-alm/observed-funding-research/src/observed_alm.py').exists()),None)
if base is None: raise FileNotFoundError('Repo root not found')
sys.path.insert(0,str(base/'04-quant-strategy-and-products/treasury-and-alm/observed-funding-research/src'))
from observed_alm import run as run_balances
from observed_funding_rates import run as run_rates
sfc=os.getenv('SFC_CANONICAL_CSV')
banrep=os.getenv('BANREP_CANONICAL_CSV')
if not sfc or not banrep:
    print('NO_DATA: authorized Drive export paths absent; no synthetic fallback.')
    study=None
else:
    dest=Path(os.environ.get('ALM_OBSERVED_OUTPUT','/tmp/alm-observed'))
    study=run_balances(sfc,dest)
    rate_study=run_rates(sfc,banrep,dest/'rate_sensitivity.json')
    print('Observed quarters:',study['observed_quarters'])
    print('Last SFC cut:',study['last'])
    print('Benchmark DTF month:',rate_study['lagged_dtf_month'])


In [ ]:
if study is None:
    print('SKIPPED: chart requires canonical observed inputs.')
else:
    import csv
    from datetime import date
    path=dest/'observed_funding_quarters.csv'
    with path.open(newline='') as f: history=list(csv.DictReader(f))
    t=[date.fromisoformat(x['fecha_corte']) for x in history]
    ratio=[float(x['ratio_cartera_depositos']) for x in history]
    fig,ax=plt.subplots(figsize=(10,4))
    ax.plot(t,ratio,label='Cartera / depósitos seleccionados')
    ax.set(ylabel='Ratio',title='Indicador descriptivo de fondeo SFC; no LCR ni NSFR')
    ax.legend();plt.show()
    scenarios=study['scenarios']
    print([(x['name'],round(100*x['assumed_total_runoff'],2),round(100*x['ratio_after_runoff'],2)) for x in scenarios])
    print('Pass-through assumptions:',rate_study['scenarios'])


## Falsación y próximos controles

- Los depósitos seleccionados no equivalen a la totalidad de fondeo bancario. El perfil de plazos, retiros reales y HQLA debe recogerse por entidad.
- Los escenarios de retiro **no** son cuantiles de pérdidas de cartera ni retiros observados.
- DTF BanRep es benchmark: no es tasa contractual pagada por todos los CDT, y el repricing del 100 % es un supuesto contrafactual.
- Para estrés comparable se requieren vintages de publicación y validación de metodologías del agregado SFC; no usar los datos revisados de hoy como datos conocidos en fechas anteriores.
- Registrar resultados y desviaciones frente a hipótesis sin sustituir datos faltantes por secuencias aleatorias.